# AION NewsImpact - Demo Notebook

**Historical News Impact Analysis using Semantic Search**

This notebook demonstrates the capabilities of the AION NewsImpact package for analyzing the potential market impact of news headlines by finding semantically similar historical headlines and their observed price effects.

## Table of Contents

1. [Setup and Installation](#Setup-and-Installation)
2. [Creating Sample Data](#Creating-Sample-Data)
3. [Basic Usage](#Basic-Usage)
4. [Query Examples](#Query-Examples)
5. [Impact Statistics](#Impact-Statistics)
6. [Advanced Features](#Advanced-Features)
7. [Integration with AION VolWeight](#Integration-with-AION-VolWeight)

## Setup and Installation

First, let's ensure the required packages are installed.

In [ ]:
# Install packages if needed (uncomment if running fresh)
# !pip install aion-newsimpact
# !pip install aion-volweight  # For integration example

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Import AION NewsImpact
from aion_newsimpact import NewsImpact, ImpactQueryResult

print("AION NewsImpact imported successfully!")
print(f"Package version: {__import__('aion_newsimpact').__version__}")

## Creating Sample Data

Let's create a sample dataset of historical news headlines with their market impacts.

In [ ]:
# Create sample historical news data
sample_data = pd.DataFrame({
    'headline': [
        'Apple reports record quarterly earnings, beats expectations',
        'Google parent Alphabet misses revenue targets',
        'Microsoft announces major AI investment',
        'Amazon stock surges on strong cloud growth',
        'Tesla faces production challenges in Q4',
        'Meta platforms cuts workforce amid restructuring',
        'Nvidia unveils new AI chip architecture',
        'Intel struggles with manufacturing delays',
        'Semiconductor shortage impacts auto industry',
        'Tech sector rallies on positive economic data',
        'Federal Reserve signals potential rate changes',
        'Market volatility increases amid trade tensions',
        'Banking sector faces regulatory scrutiny',
        'Energy stocks surge on oil price spike',
        'Healthcare stocks gain on drug approval news',
        'Consumer spending shows unexpected strength',
        'Manufacturing data indicates economic slowdown',
        'Retail earnings exceed analyst expectations',
        'Housing market shows signs of cooling',
        'Employment report beats forecasts',
    ],
    'date': [
        '2025-01-15', '2025-01-18', '2025-01-22', '2025-01-25', '2025-01-28',
        '2025-02-01', '2025-02-05', '2025-02-08', '2025-02-12', '2025-02-15',
        '2025-02-18', '2025-02-22', '2025-02-25', '2025-03-01', '2025-03-05',
        '2025-03-08', '2025-03-12', '2025-03-15', '2025-03-18', '2025-03-22',
    ],
    'returns_1d': [
        0.032, -0.028, 0.025, 0.018, -0.015,
        -0.022, 0.035, -0.012, -0.008, 0.022,
        0.008, -0.018, -0.015, 0.042, 0.028,
        0.012, -0.020, 0.015, -0.010, 0.018,
    ],
    'ticker': [
        'AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA',
        'META', 'NVDA', 'INTC', 'SOXX', 'QQQ',
        'TLT', 'SPY', 'XLF', 'XLE', 'XLV',
        'XLY', 'XLI', 'XRT', 'XHB', 'SPY',
    ],
})

print(f"Created sample dataset with {len(sample_data)} headlines")
sample_data.head(10)

## Basic Usage

Initialize the NewsImpact analyzer and run your first query.

In [ ]:
# Initialize NewsImpact analyzer
# The FAISS index is built automatically during initialization
analyzer = NewsImpact(sample_data, text_column='headline')

print(f"Analyzer initialized: {analyzer}")
print(f"Embedding dimension: {analyzer.embedding_dim}")
print(f"Index size: {analyzer.index.ntotal} vectors")

In [ ]:
# Run a simple query
query = "Company reports strong quarterly earnings"
results = analyzer.query(query, top_k=5)

print(f"Query: '{query}'")
print(f"Found {len(results)} similar headlines\n")
print(f"Average Similarity: {results.average_similarity:.3f}")
print(f"Average 1-Day Return: {results.average_return:.2%}")

In [ ]:
# Display results as a DataFrame
results_df = results.to_dataframe()
results_df

## Query Examples

Let's explore different types of queries to see how the semantic search works.

In [ ]:
# Example queries for different scenarios
queries = [
    "Earnings beat expectations",
    "Revenue misses estimates",
    "Federal Reserve interest rate",
    "Market volatility uncertainty",
    "New product announcement",
]

# Run all queries and collect results
query_results = []

for query in queries:
    results = analyzer.query(query, top_k=3)
    query_results.append({
        'Query': query,
        'Avg Similarity': results.average_similarity,
        'Avg Return': results.average_return,
        'Return Std': results.return_std,
        'Count': len(results)
    })

# Display results
pd.DataFrame(query_results)

In [ ]:
# Detailed view of a specific query
detailed_query = "Federal Reserve interest rate decision"
detailed_results = analyzer.query(detailed_query, top_k=5)

print(f"Query: '{detailed_query}'\n")
print("=" * 80)

for i, (headline, date, sim, ret, ticker) in enumerate(zip(
    detailed_results.headlines,
    detailed_results.dates,
    detailed_results.similarity_scores,
    detailed_results.returns_1d,
    detailed_results.tickers
)):
    print(f"\n{i+1}. Similarity: {sim:.3f} | {date} | {ticker}")
    print(f"   Headline: {headline}")
    print(f"   1-Day Return: {ret:+.2%}")

## Impact Statistics

Get aggregate statistics about the historical impact of news in your dataset.

In [ ]:
# Get impact statistics
stats = analyzer.get_impact_stats()

# Display statistics
print("Historical Impact Statistics")
print("=" * 50)
print(f"Total Headlines: {stats['total_headlines']}")
print(f"Date Range: {stats['date_range'][0]} to {stats['date_range'][1]}")
print()
print("Return Statistics:")
print(f"  Average 1-Day Return: {stats['avg_return_1d']:.2%}")
print(f"  Standard Deviation:   {stats['std_return_1d']:.2%}")
print(f"  Minimum Return:       {stats['min_return']:.2%}")
print(f"  Maximum Return:       {stats['max_return']:.2%}")
print()
print("Impact Distribution:")
print(f"  Positive: {stats['positive_impact_pct']:.1f}%")
print(f"  Negative: {stats['negative_impact_pct']:.1f}%")
print(f"  Neutral:  {stats['neutral_impact_pct']:.1f}%")

## Advanced Features

Explore advanced features like adding new headlines and custom models.

In [ ]:
# Adding new headlines to the index
new_headlines = pd.DataFrame({
    'headline': [
        'Tech giant announces breakthrough in quantum computing',
        'AI startup valued at $10 billion in latest funding',
        'Central bank maintains current interest rate policy',
    ],
    'date': ['2025-04-01', '2025-04-05', '2025-04-10'],
    'returns_1d': [0.045, 0.038, 0.005],
    'ticker': ['GOOGL', 'Private', 'FED'],
})

print(f"Before: {len(analyzer.historical_df)} headlines")
analyzer.add_headlines(new_headlines, rebuild_index=True)
print(f"After: {len(analyzer.historical_df)} headlines")

# Verify new headlines are searchable
results = analyzer.query('quantum computing', top_k=3)
print(f"\nQuery 'quantum computing' found {len(results)} results")
print(f"Top result: {results.headlines[0][:60]}...")

In [ ]:
# Getting embeddings for custom use
test_text = "Market analysis and predictions"
embedding = analyzer.get_embedding(test_text)

print(f"Text: '{test_text}'")
print(f"Embedding shape: {embedding.shape}")
print(f"Embedding norm: {np.linalg.norm(embedding):.6f} (should be ~1.0)")
print(f"First 10 values: {embedding[0, :10]}")

## Integration with AION VolWeight

Combine news impact analysis with VIX-based confidence adjustment.

In [ ]:
try:
    from aion_volweight import adjust_confidence, get_regime, get_regime_summary
    
    print("AION VolWeight integration available!\n")
    
    # Current market conditions
    current_vix = 18.5
    
    print(get_regime_summary(current_vix))
    print()
    
    # Analyze a news headline with volatility adjustment
    headline = "Company reports better than expected earnings"
    results = analyzer.query(headline, top_k=5)
    
    # Base confidence from similarity
    base_confidence = results.average_similarity
    
    # Adjust for volatility
    adjusted_confidence = adjust_confidence(base_confidence, current_vix)
    
    print(f"Headline: '{headline}'")
    print()
    print("Analysis Results:")
    print(f"  Similar Historical Cases: {len(results)}")
    print(f"  Base Confidence (similarity): {base_confidence:.3f}")
    print(f"  VIX-Adjusted Confidence: {adjusted_confidence:.3f}")
    print(f"  Historical Avg Return: {results.average_return:.2%}")
    print(f"  Return Volatility: {results.return_std:.2%}")
    
except ImportError:
    print("AION VolWeight not installed.")
    print("Install with: pip install aion-volweight")

## Summary

In this notebook, we demonstrated:

1. **Basic Setup**: Initializing NewsImpact with historical data
2. **Semantic Search**: Finding similar historical headlines
3. **Impact Analysis**: Retrieving observed market returns
4. **Statistics**: Getting aggregate impact statistics
5. **Advanced Features**: Adding new headlines, custom embeddings
6. **Integration**: Combining with VIX-based confidence adjustment

### Next Steps

- Load your own historical news dataset
- Experiment with different sentence-transformers models
- Integrate with your sentiment analysis pipeline
- Contribute data to the AION ecosystem

### Resources

- Documentation: https://aion-analytics.org/docs
- GitHub: https://github.com/aion-analytics/aion-newsimpact
- Data Contribution: https://aion-analytics.org/contribute